In [53]:
import pandas as pd
import numpy as np

col names:
maybe just focus on ease of getting in, cost (so feasibility of attendance), outcomes (is it worth it?/completion) and demographics/SCHOOL INFO
could not use OPEID id, had to use unit ID due to leading zeros 
MAIN
PREDDEG - filter for "3" which is predom. bachelor's degree granting
CONTROL
REGION - may as well drop, assume it doesn't matter
LOCALE - city (11-13), suburb (21-23), town (31, 33), rural (41-43)
CCUGPROF - for filtering/querying
CCSIZSET 
HBCU
WOMENONLY
ADM_RATE
SATVRMID
SATMTMID
ACTCMMID
SAT_AVG
DISTANCE_ONLY - error where it "wasn't in index?"
NPT4_PUB
NPT4_PRIV
TUITIONFEE_IN
TUITIONFEE_OUT
C150_4
RET_FT4
PCTFLOAN
COUNT_NWNE_P10
COUNT_WNE_P10
MD_EARN_WNE_P10
PCT25_EARN_WNE_P10
PCT75_EARN_WNE_P10
TRANS_4
ICLEVEL - only 4 year (1)
UGDS_MEN
UGDS_WOMEN
D_PCTPELL_PCTFLOAN
PRGMOFR
MDCOMP_ALL
PCTPELL_DCS
PCTFLOAN_DCS

In [70]:
edu_all = pd.read_csv("Most-Recent-Cohorts-Institution.csv", index_col='UNITID').fillna(0)
#filter for schools for four year degrees, only public or private non-profit, only four year colleges 
edu_all = edu_all[edu_all['ICLEVEL'] == 1]
edu_all = edu_all[(edu_all['CONTROL'] == 1) | (edu_all['CONTROL'] == 2)]
edu_all = edu_all.query('CCUGPROF >= 5')

edu = edu_all[["MAIN","CONTROL","LOCALE", "HBCU","WOMENONLY","ADM_RATE","ACTCMMID","SAT_AVG","NPT4_PUB","NPT4_PRIV","TUITIONFEE_IN","TUITIONFEE_OUT","C150_4","RET_FT4","PCTFLOAN","COUNT_NWNE_P10","COUNT_WNE_P10","MD_EARN_WNE_P10","PCT25_EARN_WNE_P10","PCT75_EARN_WNE_P10","TRANS_4","UGDS_MEN","UGDS_WOMEN","D_PCTPELL_PCTFLOAN","PCTPELL_DCS","PCTFLOAN_DCS"]]

C:\Users\gabri\AppData\Local\Temp\ipykernel_28752\3634495635.py:1: DtypeWarning: Columns (9,1537,1540,1542,1606,1608,1614,1615,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1690,1692,1697,1700,1725,1726,1727,1728,1729,1743,1815,1816,1817,1818,1823,1824,1830,1831,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1909,1910,1911,1912,1913,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1983,1984,2376,2377,2403,2404,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2958,3215,3231,3235,3236) have mixed types. Specify dtype option on import or set low_memory=False.
  edu_all = pd.read_csv("Most-Recent-Cohorts-Institution.csv", index_col='UNITID').fillna(0)


In [71]:
#one-hot encoding for locale:
edu[['CITY', 'SUBURB', 'TOWN', 'RURAL']] = 0

#iterate through each row. if locale == certain number, then reassign to 1, else skip 
for i, r in edu.iterrows():
    if 11 <= r["LOCALE"] <= 13:
        edu.loc[i, "CITY"] = 1
    elif 21 <= r["LOCALE"] <= 23:
        edu.loc[i,"SUBURB"] = 1
    elif 31 <= r["LOCALE"] <= 33:
        edu.loc[i,"TOWN"] = 1
    elif 41 <= r["LOCALE"] <= 43:
        edu.loc[i,"RURAL"] = 1

C:\Users\gabri\AppData\Local\Temp\ipykernel_28752\350535846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edu[['CITY', 'SUBURB', 'TOWN', 'RURAL']] = 0
C:\Users\gabri\AppData\Local\Temp\ipykernel_28752\350535846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edu[['CITY', 'SUBURB', 'TOWN', 'RURAL']] = 0
C:\Users\gabri\AppData\Local\Temp\ipykernel_28752\350535846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] 

In [72]:
#change control to 0 and 1 rather than 1 and 2
edu = edu.rename(columns={'CONTROL':'PRIVATE'})
for i, r in edu.iterrows():
    if r["PRIVATE"] == 1:
        edu.loc[i,"PRIVATE"] = 0
        edu.loc[i,"NET_PRICE"] = edu.loc[i, "NPT4_PUB"]
    elif r["PRIVATE"] == 2:
        edu.loc[i,"PRIVATE"] = 1
        edu.loc[i,"NET_PRICE"] = edu.loc[i, "NPT4_PRIV"]
#error: sometimes NET PRICE doesn't take the NPT4_PRIV value, usually just run it again from the start and don't run this cell otherwise
edu_cleaner = edu.drop(edu[edu['NET_PRICE'] == 0].index)

In [73]:
#drop NPT4 pub and priv, locale
edu_cleaner = edu_cleaner.drop(['NPT4_PUB', 'NPT4_PRIV', 'LOCALE'], axis=1)

edu_cleaner

,MAIN,PRIVATE,HBCU,WOMENONLY,ADM_RATE,ACTCMMID,SAT_AVG,TUITIONFEE_IN,TUITIONFEE_OUT,C150_4,...,UGDS_MEN,UGDS_WOMEN,D_PCTPELL_PCTFLOAN,PCTPELL_DCS,PCTFLOAN_DCS,CITY,SUBURB,TOWN,RURAL,NET_PRICE
UNITID,,,,,,,,,,,,,,,,,,,,,
100654,1,0,1.0,0.0,0.6840,18.0,920.0,10024.0,18634.0,0.2678,...,0.4055,0.5945,5107.0,0.6553,0.5365,1,0,0,0,14982.0
100663,1,0,0.0,0.0,0.8668,27.0,1291.0,8832.0,21216.0,0.6442,...,0.3752,0.6248,13547.0,0.3374,0.4214,1,0,0,0,16755.0
100706,1,0,0.0,0.0,0.7810,28.0,1259.0,11878.0,24770.0,0.6295,...,0.5981,0.4019,7569.0,0.2235,0.3511,1,0,0,0,18240.0
100724,1,0,1.0,0.0,0.9660,18.0,963.0,11068.0,19396.0,0.2773,...,0.3595,0.6405,3499.0,0.6984,0.7680,1,0,0,0,13527.0
100751,1,0,0.0,0.0,0.8006,26.0,1304.0,11940.0,32300.0,0.7276,...,0.4399,0.5601,31685.0,0.1843,0.3494,1,0,0,0,20888.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494630,1,1,0.0,0.0,0.0000,0.0,0.0,8229.0,8229.0,0.4375,...,0.5532,0.4468,44.0,0.5682,0.0000,0,1,0,0,8757.0
494685,1,1,0.0,0.0,0.8529,0.0,1039.0,7420.0,7420.0,0.5714,...,0.4047,0.5953,445.0,0.3846,0.4367,0,1,0,0,9006.0
494737,1,1,0.0,0.0,0.0000,0.0,0.0,20000.0,20000.0,0.2222,...,1.0000,0.0000,177.0,0.8701,0.0000,1,0,0,0,7174.0


In [ ]:
edu_cleaner.index = edu_cleaner.index.astype(str)

In [76]:
from scipy.sparse import lil_matrix
import scipy.spatial.distance

#target school = UMD
target_school_id = '163286'
target_school = edu_cleaner.loc[target_school_id]
distances = scipy.spatial.distance.cdist(edu_cleaner, [target_school], metric='euclidean').flatten()
query_distances = list(zip(edu_cleaner.index, distances))

for similar_school_id, similar_score in sorted(query_distances, key=lambda x:x[1], reverse=False)[:10]:
    print(similar_school_id, similar_score)

163286 0.0
110653 7869.468031840913
233921 8289.208650789107
145637 8832.96428287181
110644 10500.668076521239
236948 10746.693771768198
110662 11666.255233166352
240444 13391.65669402435
232186 13494.860810895027
110680 13621.433664195483
